In [9]:
import os
import re
import glob
import argparse
import requests
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from urllib.parse import urlencode
from datetime import datetime
from rasterstats import zonal_stats

In [ ]:
#-- Get all years of annual ranks from 1990- Rainfall---

#tif path to annual rainfall tifs (1990-)
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/rf_all/annual"

all_records = []
for tif in sorted(glob.glob(os.path.join(tif_path, f"*.tif"))):
    parts = os.path.basename(tif).replace(".tif", "").split("_")
    year = parts[1]
    date = f"{year}"

    with rasterio.open(tif) as src:
        arr = src.read(1).astype(float)
        arr = np.where(arr == src.nodata, np.nan, arr)
        mean_val = np.nanmean(arr)

    all_records.append({
        "date": date,
        "year": int(year),
        "mean": mean_val
    })

df = pd.DataFrame(all_records)

df["rank"] = df["mean"].rank(method="min", ascending=True)

df

,date,year,mean,rank
0,1920,1920,1546.811332,33.0
1,1921,1921,1904.286918,75.0
2,1922,1922,1799.935181,68.0
3,1923,1923,2512.253361,105.0
4,1924,1924,1638.628186,47.0
...,...,...,...,...
101,2021,2021,1778.105613,63.0
102,2022,2022,1128.450986,4.0
103,2023,2023,1589.244796,41.0
104,2024,2024,1419.567413,20.0


In [ ]:
#--Get all months of this year's ranks from 1990- Rainfall --

tif_path = "/Users/cherryleheu/Documents/HCDP/Data/rf_all/monthly"

for m in np.arange(1,13):
    all_records = []
    for tif in sorted(glob.glob(os.path.join(tif_path, f"*{m:02d}.tif"))):
        parts = os.path.basename(tif).replace(".tif", "").split("_")
        year = parts[1]
        date = f"{year}"

        with rasterio.open(tif) as src:
            arr = src.read(1).astype(float)
            arr = np.where(arr == src.nodata, np.nan, arr)
            mean_val = np.nanmean(arr)

        all_records.append({
            "date": date,
            "year": int(year),
            "mean": mean_val
        })

    df = pd.DataFrame(all_records)

    df["rank"] = df["mean"].rank(method="min", ascending=True)

    print(f"For month: {m}: {df[df['date']=='2025']['rank'].values}")


For month: 1: [56.]
For month: 2: [4.]
For month: 3: [31.]
For month: 4: [25.]
For month: 5: [21.]
For month: 6: [51.]
For month: 7: [16.]
For month: 8: [3.]
For month: 9: [23.]
For month: 10: [21.]
For month: 11: [40.]
For month: 12: [14.]


In [ ]:
#-- Get all years of annual ranks from 1990- Tmean---
#Tif path to annual tmean tifs (1990-)
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/annual/tmean/calendar_year"
climo_path = "/Users/cherryleheu/Documents/HCDP/Data/climo/30yr/monthly_tmean_statewide_1991-2020_annual.tif"

all_records = []

with rasterio.open(climo_path) as src:
    clim = src.read(1, masked=True).astype("float64")
    climo_mean = clim.mean()
    climo_f = climo_mean*9/5 + 32

for tif in sorted(glob.glob(os.path.join(tif_path, f"*_agg.tif"))):
    parts = os.path.basename(tif).replace(".tif", "").split("_")
    year = parts[2]
    date = f"{year}"

    with rasterio.open(tif) as src:
        arr = src.read(1).astype(float)
        arr = np.where(arr == src.nodata, np.nan, arr)
        mean_val = np.nanmean(arr)
        mean_val_f = mean_val*9/5 + 32

    all_records.append({
        "date": date,
        "year": int(year),
        "mean": mean_val_f,
        "anom": mean_val_f - climo_f
    })

df = pd.DataFrame(all_records)

df["rank"] = df["mean"].rank(method="min", ascending=False)

df

,date,year,mean,anom,rank
0,1990,1990,65.801174,-0.492514,28.0
1,1991,1991,65.641067,-0.652622,32.0
2,1992,1992,65.888010,-0.405678,27.0
3,1993,1993,65.107939,-1.185750,35.0
4,1994,1994,65.549973,-0.743716,33.0
5,1995,1995,66.172545,-0.121143,22.0
6,1996,1996,66.240123,-0.053566,20.0
7,1997,1997,65.703645,-0.590043,30.0
8,1998,1998,65.045232,-1.248457,36.0
9,1999,1999,65.273941,-1.019748,34.0


In [ ]:
# -- Get average value, p_anomaly, diff_anomaly, ranks for rainfall for each island from `1920-`--

#County shapefile
shapefile = "../../public/shapefiles/islands.shp"
# raster_folder = "/Users/cherryleheu/Documents/HCDP/Data/monthly/rainfall/"
climo_file = "/Users/cherryleheu/Documents/HCDP/Data/climo/30yr/monthly_rainfall_clim_statewide_1991-2020_annual.tif"
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/rf_all/annual"
gdf = gpd.read_file(shapefile).dissolve(by="name", as_index=False).reset_index(drop=True)

climo_stats = zonal_stats(gdf, climo_file, stats=["mean"])
gdf["climo_mean"] = [(c["mean"] / 25.4) if c["mean"] is not None else np.nan for c in climo_stats]

all_records = []

for tif in sorted(glob.glob(os.path.join(tif_path, "*.tif"))):
    name = os.path.basename(tif).replace(".tif", "")
    parts = name.split("_")
    year = parts[1] 
    with rasterio.open(tif) as src:
        nodata = src.nodata

    stats = zonal_stats(gdf, tif, stats=["mean"], nodata=nodata)

    for i, division in gdf.iterrows():
        mean_raw = stats[i]["mean"]
        climo_mean = division["climo_mean"]

        if mean_raw is None or climo_mean is None or climo_mean == 0 or np.isnan(climo_mean):
            mean_val = np.nan
            anomaly = np.nan
        else:
            mean_val = mean_raw / 25.4
            anomaly = (mean_val - climo_mean) / climo_mean * 100

        all_records.append({
            "island": division["name"],
            "date": int(year),
            "mean": mean_val,
            "p_anomaly": f"{round(anomaly)}%",
            "diff_anomaly": mean_val - climo_mean,
        })

df = pd.DataFrame(all_records)
df["rank"] = df.groupby("island")["mean"].rank(method="min", ascending=True).astype("Int64")
print(df[df['date'] == 2025])
df[df['date'] == 2025].to_csv("island_annual_rainfall_2025_ranks.csv", index=False)

        island  date       mean p_anomaly  diff_anomaly  rank
735     Hawaii  2025  40.653415      -36%    -22.937288     2
736  Kahoolawe  2025   9.918075      -14%     -1.666635    23
737      Kauai  2025  67.016988      -11%     -8.386554    21
738      Lanai  2025  12.761269      -28%     -4.961560    15
739       Maui  2025  40.665760      -41%    -27.868129     1
740    Molokai  2025  25.554383      -38%    -15.751622     5
741       Oahu  2025  46.042890      -15%     -8.173708    16


In [ ]:
# -- Get average value, diff_anomaly, ranks for tmean for each island from `1990-`--

#County shapefile
shapefile = "../../public/shapefiles/islands.shp"

climo_file = "/Users/cherryleheu/Documents/HCDP/Data/climo/30yr/monthly_tmean_statewide_1991-2020_annual.tif"
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/annual/tmean/calendar_year"

gdf = gpd.read_file(shapefile).dissolve(by="name", as_index=False).reset_index(drop=True)

climo_stats = zonal_stats(gdf, climo_file, stats=["mean"])
gdf["climo_mean"] = [(c["mean"] * 1.8 + 32) if c["mean"] is not None else np.nan for c in climo_stats]

all_records = []

for tif in sorted(glob.glob(os.path.join(tif_path, "*_agg.tif"))):
    name = os.path.basename(tif).replace(".tif", "")
    parts = name.split("_")
    year = parts[2] 

    with rasterio.open(tif) as src:
        nodata = src.nodata

    stats = zonal_stats(gdf, tif, stats=["mean"], nodata=nodata)

    for i, division in gdf.iterrows():
        mean_raw = stats[i]["mean"]
        climo_mean = division["climo_mean"]

        if mean_raw is None or climo_mean is None or climo_mean == 0 or np.isnan(climo_mean):
            mean_val = np.nan
            anomaly = np.nan
        else:
            mean_val = mean_raw * 1.8 + 32
            anomaly = (mean_val - climo_mean) 

        all_records.append({
            "island": division["name"],
            "date": int(year),
            "mean": mean_val,
            "value": f"{mean_val:.2f} ({anomaly:+.2f})",
        })

tmeandf = pd.DataFrame(all_records)
tmeandf["rank"] = tmeandf.groupby("island")["mean"].rank(method="min", ascending=False).astype("Int64")

print(tmeandf[tmeandf['date'] == 2025])
tmeandf[tmeandf['date'] == 2025].to_csv("island_annual_tmean_2025_tmean_ranks.csv", index=False)

        island  date       mean          value  rank
245     Hawaii  2025  63.975182  63.98 (+0.69)     8
246  Kahoolawe  2025  74.937319  74.94 (+0.89)     4
247      Kauai  2025  71.795279  71.80 (+0.91)     3
248      Lanai  2025  73.490867  73.49 (+0.86)     4
249       Maui  2025  69.666219  69.67 (+0.78)     3
250    Molokai  2025  73.468817  73.47 (+0.86)     4
251       Oahu  2025  74.491756  74.49 (+1.01)     4


In [10]:
import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
import rasterio
from rasterstats import zonal_stats

tif_path = "/Users/cherryleheu/Documents/HCDP/Data/rf_all/monthly"

shapefile = "../../public/shapefiles/islands.shp"
gdf = gpd.read_file(shapefile).dissolve(by="name", as_index=False).reset_index(drop=True)

print(gdf["name"].unique())  # inspect names first

oahu = gdf[gdf["name"] == "Oahu"]   # change string if needed

for m in range(1, 13):
    all_records = []

    for tif in sorted(glob.glob(os.path.join(tif_path, f"*{m:02d}.tif"))):
        parts = os.path.basename(tif).replace(".tif", "").split("_")
        year = int(parts[1])

        with rasterio.open(tif) as src:
            stats = zonal_stats(
                oahu,
                tif,
                stats=["mean"],
                nodata=src.nodata
            )

        mean_val = stats[0]["mean"]

        all_records.append({
            "year": year,
            "mean": mean_val
        })

    df = pd.DataFrame(all_records).sort_values("year")

    df["rank"] = df["mean"].rank(method="min", ascending=True)

    print(f"Month {m}: 2025 rank =", df.loc[df["year"] == 2025, "rank"].values[0])

['Hawaii' 'Kahoolawe' 'Kauai' 'Lanai' 'Maui' 'Molokai' 'Oahu']
Month 1: 2025 rank = 77.0
Month 2: 2025 rank = 9.0
Month 3: 2025 rank = 28.0
Month 4: 2025 rank = 80.0
Month 5: 2025 rank = 67.0
Month 6: 2025 rank = 34.0
Month 7: 2025 rank = 38.0
Month 8: 2025 rank = 1.0
Month 9: 2025 rank = 12.0
Month 10: 2025 rank = 17.0
Month 11: 2025 rank = 26.0
Month 12: 2025 rank = 76.0


In [12]:
import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
import rasterio
from rasterstats import zonal_stats

tif_path = "/Users/cherryleheu/Documents/HCDP/Data/monthly/tmean"

shapefile = "../../public/shapefiles/islands.shp"
gdf = gpd.read_file(shapefile).dissolve(by="name", as_index=False).reset_index(drop=True)

print(gdf["name"].unique())  # inspect names first

oahu = gdf[gdf["name"] == "Oahu"]   # change string if needed

for m in range(1, 13):
    all_records = []

    for tif in sorted(glob.glob(os.path.join(tif_path, f"*{m:02d}.tif"))):
        parts = os.path.basename(tif).replace(".tif", "").split("_")
        year = int(parts[1])

        with rasterio.open(tif) as src:
            stats = zonal_stats(
                oahu,
                tif,
                stats=["mean"],
                nodata=src.nodata
            )

        mean_val = stats[0]["mean"]

        all_records.append({
            "year": year,
            "mean": mean_val
        })

    df = pd.DataFrame(all_records).sort_values("year")

    df["rank"] = df["mean"].rank(method="min", ascending=False)

    print(f"Month {m}: 2025 rank =", df.loc[df["year"] == 2025, "rank"].values[0])

['Hawaii' 'Kahoolawe' 'Kauai' 'Lanai' 'Maui' 'Molokai' 'Oahu']
Month 1: 2025 rank = 7.0
Month 2: 2025 rank = 4.0
Month 3: 2025 rank = 3.0
Month 4: 2025 rank = 1.0
Month 5: 2025 rank = 8.0
Month 6: 2025 rank = 16.0
Month 7: 2025 rank = 13.0
Month 8: 2025 rank = 9.0
Month 9: 2025 rank = 8.0
Month 10: 2025 rank = 8.0
Month 11: 2025 rank = 9.0
Month 12: 2025 rank = 5.0


In [13]:
df

,year,mean,rank
0,1990,21.115075,32.0
1,1991,21.946277,21.0
2,1992,22.317570,10.0
3,1993,21.432044,29.0
4,1994,21.489105,27.0
5,1995,22.518446,7.0
6,1996,20.446077,36.0
7,1997,20.873174,34.0
8,1998,21.426780,30.0
9,1999,21.485435,28.0
